In [ ]:
import os, glob
from collections import OrderedDict

import pandas as pd
import numpy as np
from scipy.stats import sem
import matplotlib.pyplot as plt

from flow_analysis_tools.color_palettes import *
from flow_analysis_tools import flow_violin

plt.style.use("ggplot")

## Time course analysis for pre-processed data
Assumes that the data from multiple timepoints has already been processed in `initial_violin_analysis.ipynb`.

Assumes that data will be in a root directory, contain desired keywords, and have time points as "rXdY". Example:

```
ROOT_DIR
|--YYMMDD-<KEYWORD_A>-r1d1
|--|--processed_data.csv
|--YYMMDD-<KEYWORD_B>-r1d1
|--|--processed_data.csv
|--YYMMDD-<KEYWORD_A>-r1d2
|--|--processed_data.csv
|--YYMMDD-<KEYWORD_B>-r1d2
|--|--processed_data.csv
```

If `KEYWORD` variable below is set to "KEYWORD_A", then only those files will be analyzed, an a 2-day timecourse would be created.

##### Variables for Finding the Data

In [ ]:
ROOT_DIR = '/Users/alexandresathler/Library/CloudStorage/Box-Box/HsiungLab/arsathler/Flow Results/2602-LX2-HEKCLTA-RENDER-Rep1'
KEYWORD = "LX2_RENDER"
DATA_FILE_NAME = "processed_data.csv"
WORD_SEPARATOR = "_"
IDX_WORDS_TO_DAY = 3 # what word (separated by WORD_SEPARATOR) is the word containing the day? Include the index.
DAY_SEPARATOR = "d"
YLIMITS = (0, 7)
DAYS_DESIRED = [3, 6, 14]

##### Variables for Plotting the Data

In [ ]:
KEY_COLUMNS = ["Construct", "Guide"]
PLOT_COLUMN = "RL1-A"

In [ ]:
CENTER_STATISTIC = "median" # choose mean or median
RANGE_STATISTIC = "SEM" # choose between STD, IQR, and SEM
assert CENTER_STATISTIC in ["mean", "median"]
assert RANGE_STATISTIC in ["STD", "IQR", "SEM"]

### Loading Data

In [ ]:
days = []
dfs = []
for root, dirs, files in os.walk(ROOT_DIR):
    for d in dirs:
        if KEYWORD not in d:
            continue
        this_data_filepath = os.path.join(root, d, DATA_FILE_NAME)
        this_day = int(d.split(WORD_SEPARATOR)[IDX_WORDS_TO_DAY].split(DAY_SEPARATOR)[1])
        if DAYS_DESIRED is not None and this_day not in DAYS_DESIRED: 
            print(f"Found day {this_day}, but not among desired days; skipping...")
            continue
        if not os.path.isfile(this_data_filepath): raise ValueError(f"Data file '{DATA_FILE_NAME}' not found at {os.path.join(root, d)}")
        days.append(this_day)
        dfs.append(pd.read_csv(this_data_filepath))
if len(days) == 0: raise ValueError(f"No valid DFs found at root {ROOT_DIR}")
print(days)

In [ ]:
for day, df in zip(days, dfs):
    df["Day"] = np.full(shape=(df.shape[0]), fill_value=day)

# from: https://stackoverflow.com/questions/23668427/pandas-three-way-joining-multiple-dataframes-on-columns
data_df = pd.concat(dfs).drop("Unnamed: 0", axis=1)

In [ ]:
for key_col in KEY_COLUMNS:
    assert key_col in data_df.columns, f"Could not find {key_col} in data."
    print(f"{key_col}: {data_df[key_col].unique()}")
data_df.head()

### Compiling Statistics

In [ ]:
centers = {}
ranges = {}
day_0_value = np.median(
    data_df[
        (data_df["Single_Cell"] == True) \
        & (data_df["Day"] == 14) \
        & (data_df["Construct"] == "LX-2 (TGFß)") \
        & (data_df["Guide"] == "αCol1A1") \
    ][PLOT_COLUMN]
)

for const in ["1X", "20X"]:
    for guide in ["sgNT", "sgCol1A1"]:
        these_centers = [np.log10(day_0_value)]
        these_ranges = [0]
        for day in sorted(days):
            these_data = data_df[
                (data_df["Single_Cell"] == True) \
                & (data_df["Day"] == day) \
                & (data_df["Construct"] == const) \
                & (data_df["Guide"] == guide)
            ]
            these_data.loc[:,PLOT_COLUMN] = np.log10(these_data[PLOT_COLUMN])
            these_data = these_data.fillna(0)
            these_data_col = these_data[PLOT_COLUMN]
            these_centers.append(np.median(these_data_col))
            these_ranges.append(sem(these_data_col))
        centers[f"{guide}_{const}"] = these_centers
        ranges[f"{guide}_{const}"] = these_ranges

    

In [ ]:
centers

### Graphing

In [ ]:
days.append(0)

In [ ]:
this_palette = np.array(black_red_pastel_palette)
palette = np.stack([this_palette[1], this_palette[1] - 0.25, this_palette[0], this_palette[0] - 0.25])

In [ ]:
fig, axs = plt.subplot_mosaic(
    [
        ["Control", "Timecourse"]
    ],
    figsize=(12,6),
    sharey=True,
    width_ratios=(1,3)
)

flow_violin(
    data=data_df[(data_df["Single_Cell"] == True) & (data_df["Day"] == 14) & \
                 ((data_df["Construct"] == "LX-2") | (data_df["Construct"] == "LX-2 (TGFß)"))],
    # hue="Guide",
    color=black_red_pastel_palette[1],
    x="Construct",
    y=PLOT_COLUMN,
    split=True,
    hue="Guide",
    hue_order=["αCol1A1", "Unlabelled"],
    order=["LX-2", "LX-2 (TGFß)"],
    ylabel="Col1A1 (AF647)",
    ax=axs["Control"],
    log_y_axis=True,
    xlabel="Control",
    palette=black_red_pastel_palette
)

for idx, cond in enumerate(sorted(centers.keys())):
    axs["Timecourse"].errorbar(
        x=sorted(days), y=centers[cond],
        yerr=ranges[cond],
        # label=cond,
        color=palette[idx,:],
        zorder=1
    )

for idx, cond in enumerate(sorted(centers.keys())):
    axs["Timecourse"].scatter(
        x=sorted(days), y=centers[cond],
        label=cond,
        color=np.clip(np.array(palette)[idx,:]-0.25, a_min=0, a_max=1),
        zorder=2
    )

# ax.legend(sorted(ax.get_legend_handles_labels()[1]))
axs["Control"].legend(loc="upper right")
axs["Control"].set_ybound(*YLIMITS)
axs["Timecourse"].set_xlabel("Days Post-Delivery")
axs["Timecourse"].legend(loc = "lower right")
plt.savefig(os.path.join(ROOT_DIR, "LX2_AF647_Timecourse.png"))